**Module 7 Lab 3 — Titanic survival prediction (Problem 8.1)**

Repo: https://github.com/washingtonjustin02smc/CS82A-repository/tree/module7

The dataset is the standard 891-row Titanic training file. Put `titanic_train.csv` in the same folder as this notebook. The separate unlabeled Titanic test file is not used for accuracy.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv("titanic_train.csv")
print("Rows, columns:", df.shape)
print("Missing ages:", df["Age"].isna().sum())
print("Survival label counts:")
print(df["Survived"].value_counts().sort_index().to_string())

Rows, columns: (891, 12)
Missing ages: 177
Survival label counts:
Survived
0    549
1    342


**Feature preparation:** Use passenger class (`Pclass`), age (`Age`), encoded sex (`SexFemale`), and ticket fare (`Fare`). Female is 1 and male is 0. Age is filled with the median learned **only on the training portion**, avoiding information leaking from test to train. No survival-derived field is included.

In [2]:
prepared = df.copy()
prepared["SexFemale"] = prepared["Sex"].map({"male": 0, "female": 1})
features = ["Pclass", "Age", "SexFemale", "Fare"]
X = prepared[features]
y = prepared["Survived"]
print("Features:", features)
print("Missing values before imputation:")
print(X.isna().sum().to_string())
print("Sex encoding examples:")
print(prepared[["Sex", "SexFemale"]].drop_duplicates().to_string(index=False))

Features: ['Pclass', 'Age', 'SexFemale', 'Fare']
Missing values before imputation:
Pclass         0
Age          177
SexFemale      0
Fare           0
Sex encoding examples:
   Sex  SexFemale
  male          0
female          1


**Honest evaluation:** Hold out 30% once, using `random_state=42` and `stratify=y` to preserve the survival proportions. A shallow decision tree (`max_depth=3`) is trained only on the other 70%.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeClassifier(max_depth=3, random_state=42)
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print(f"TEST accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("Confusion matrix (rows: actual 0, 1; columns: predicted 0, 1):")
print(confusion_matrix(y_test, predictions))

Training rows: 623
Test rows: 268
TEST accuracy: 0.7948 (79.48%)
Confusion matrix (rows: actual 0, 1; columns: predicted 0, 1):
[[156   9]
 [ 46  57]]


**Error analysis:** Count incorrect predictions by sex and passenger class *in the held-out set*. That tells us which specific group produces the largest number of errors, rather than guessing from the training results.

In [4]:
errors = X_test.copy()
errors["Actual"] = y_test
errors["Predicted"] = predictions
errors["Incorrect"] = errors["Actual"] != errors["Predicted"]
errors["Sex"] = errors["SexFemale"].map({0: "male", 1: "female"})
by_group = (errors.groupby(["Sex", "Pclass"])["Incorrect"]
            .agg(Mistakes="sum", Passengers="count")
            .sort_values(["Mistakes", "Passengers"], ascending=False))
print("Held-out test errors by group:")
print(by_group.to_string())
worst_sex, worst_class = by_group.index[0]
worst_count = int(by_group.iloc[0]["Mistakes"])
print(f"\nOne-sentence finding: The model most often gets {worst_sex} passengers "
      f"in class {worst_class} wrong ({worst_count} mistakes in the held-out set).")

Held-out test errors by group:
               Mistakes  Passengers
Sex    Pclass                      
female 3             20          44
male   3             15         101
       1             15          41
       2              3          29
female 1              1          28
       2              1          25

One-sentence finding: The model most often gets female passengers in class 3 wrong (20 mistakes in the held-out set).


**Conclusion:** This model uses four straightforward passenger features to predict survival. The reported accuracy comes only from the held-out 30%, not from the unlabeled `titanic_test.csv`. The error-group counts describe this particular split, not necessarily every future sample.

**Source note:** `titanic_train.csv` is the standard Kaggle Titanic 891-passenger training data (also distributed publicly in the DataScienceDojo Titanic dataset).

*Before submitting: review the feature choices and rewrite the one-sentence error interpretation in your own words, as required by the course AI-use policy.*